# Issue #382 — cutoff-driven learning evidence
This notebook reads the source-bound package only. It demonstrates the real mapping pilot and shared temporal receipt; it does not tune a betting strategy.

In [1]:
from pathlib import Path
import json
import pandas as pd

root = Path.cwd()
while not (root / 'results/cutoff_mapping').is_dir():
    root = root.parent
package = root / 'results/cutoff_mapping'
receipt = json.loads((package / 'mapping_receipt.json').read_text())
decision = json.loads((package / 'decision.json').read_text())
parity = pd.read_parquet(package / 'accepted_354_parity.parquet')
weights = pd.read_parquet(package / 'weight_ledger.parquet')
pd.DataFrame([{
    'cutoff': receipt['cutoff_identity'],
    'max_label_ts': receipt['maximum_training_label_timestamp'],
    'eligible_rows': receipt['partition']['final_refit_rows'],
    'ess': receipt['weight_receipt']['effective_sample_size'],
    'mapping_accepted': receipt['mapping_accepted'],
    'parity_rows': len(parity),
    'exact_parity_rows': int(parity['exact_354'].sum()),
}])

,cutoff,max_label_ts,eligible_rows,ess,mapping_accepted,parity_rows,exact_parity_rows
0,EVALUATION_YEAR_2022,2021-12-31T09:00:00+00:00,136738,136738.0,True,48658,48658


In [2]:
metrics = pd.DataFrame(receipt['metrics']).T
metrics[['experimental_minus_control_log_loss', 'experimental_minus_control_brier']]

,experimental_minus_control_log_loss,experimental_minus_control_brier
calibration,-0.014820,-0.011008
tune,-0.014065,-0.011180


In [3]:
assert weights['weight'].sum() == receipt['weight_receipt']['sum']
assert decision['equivalent_cutoffs_same_science']
assert parity['exact_354'].all()
decision

{'cache_keys_cutoff_bound': True,
 'contract': 'BJ-UNIFIED-CUTOFF-LEARNING-001/v2',
 'equivalent_cutoffs_same_science': True,
 'exact_354_rows': 48658,
 'future_consumers_must_pass_cutoff_explicitly': True,
 'mapping_accepted': True,
 'parity_rows': 48658,
 'verdict': 'CUTOFF_MAPPING_INTERFACE_READY'}